# 01 — Data Collection & Preprocessing
## ML-Enhanced Portfolio Construction: Volatility Forecasting & Risk Estimation
### *Niraj Mhatre | MSc Statistics | IIT Kanpur*

**Universe:** S&P 100 constituents | **Period:** 2014–2025 | **Frequency:** Daily

---
**Pipeline Position:** `Data Collection → Cleaning → Return Engineering → Quality Checks`

> *"In quantitative finance, the quality of your data pipeline determines the ceiling of every model downstream."*


## 0. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import jarque_bera, shapiro, kurtosis, skew
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.stats.stattools import durbin_watson

import yfinance as yf
import pandas_datareader.data as web
from datetime import datetime, timedelta
import os, pickle

# ── Plot style ────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'figure.facecolor': 'white',
})

SEED = 42
np.random.seed(SEED)

# ── Colour palette (quant finance style) ──────────────────────────────────────
COLORS = {
    'primary'  : '#003087',   # deep navy
    'secondary': '#C6002B',   # signal red
    'accent'   : '#F5A623',   # amber
    'positive' : '#1a7a4a',
    'negative' : '#C6002B',
    'neutral'  : '#5b5b5b',
}

print("All imports loaded successfully.")
print(f"Pandas {pd.__version__} | NumPy {np.__version__}")


## 1. Universe Definition — S&P 100

We use the **S&P 100** as our investment universe for the following reasons:

- **Liquidity**: All constituents are highly liquid — no bid-ask spread concerns
- **Data availability**: Complete price history from 2014 on Yahoo Finance
- **Survivorship bias note**: We use the *current* S&P 100 composition (2025 snapshot).
  This introduces survivorship bias — we acknowledge this as a limitation and note that
  a production system would use point-in-time index membership. For academic purposes, 
  this is standard practice and should be disclosed in any report.
- **Interview angle**: Interviewers respect candidates who flag survivorship bias proactively.


In [ ]:
# S&P 100 tickers — 2025 snapshot (current constituents)
# Source: https://en.wikipedia.org/wiki/S%26P_100
SP100_TICKERS = [
    'AAPL','ABBV','ABT','ACN','ADBE','AIG','AMD','AMGN','AMT','AMZN',
    'AVGO','AXP','BA','BAC','BK','BKNG','BLK','BMY','BRK-B','C',
    'CAT','CHTR','CL','CMCSA','COF','COP','COST','CRM','CSCO','CVS',
    'CVX','DE','DHR','DIS','DOW','DUK','EMR','EXC','F','FDX',
    'GD','GE','GILD','GM','GOOG','GOOGL','GS','HD','HON','IBM',
    'INTC','INTU','JNJ','JPM','KHC','KMI','KO','LIN','LLY','LMT',
    'LOW','MA','MCD','MDLZ','MDT','MET','META','MMM','MO','MRK',
    'MS','MSFT','NEE','NFLX','NKE','NVDA','ORCL','PEP','PFE','PG',
    'PM','PYPL','QCOM','RTX','SBUX','SCHW','SLB','SO','SPG','T',
    'TGT','TMO','TSLA','TXN','UNH','UNP','UPS','USB','V','VZ',
    'WBA','WFC','WMT','XOM',
]

MARKET_TICKERS = ['^GSPC', '^VIX']   # S&P500 index + VIX

START_DATE = '2014-01-01'
END_DATE   = '2025-01-01'
DATA_DIR   = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Universe: {len(SP100_TICKERS)} stocks")
print(f"Period  : {START_DATE} → {END_DATE}")
print(f"Tickers : {SP100_TICKERS[:10]} ...")


## 2. Data Download

In [ ]:
def download_price_data(tickers, start, end, cache_path='data/raw_prices.pkl'):
    """
    Download adjusted close prices from Yahoo Finance.
    Uses caching so you don't re-download on every run.
    
    Parameters
    ----------
    tickers    : list of ticker strings
    start, end : date strings 'YYYY-MM-DD'
    cache_path : path to cache pickle
    
    Returns
    -------
    DataFrame: daily adjusted close prices (Date × Ticker)
    """
    if os.path.exists(cache_path):
        print(f"Loading cached data from {cache_path} ...")
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    print(f"Downloading {len(tickers)} tickers from Yahoo Finance ...")
    print("This may take 2-5 minutes on first run.")

    raw = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,   # adjusts for splits and dividends automatically
        progress=True,
        group_by='ticker',
        threads=True,
    )

    # Extract Adjusted Close prices
    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw.xs('Close', axis=1, level=1)
    else:
        prices = raw[['Close']]
        prices.columns = tickers[:1]

    with open(cache_path, 'wb') as f:
        pickle.dump(prices, f)
    
    print(f"Downloaded and cached: {prices.shape}")
    return prices


def download_market_data(start, end, cache_path='data/market_data.pkl'):
    """Download S&P500 index (^GSPC) and VIX (^VIX)."""
    if os.path.exists(cache_path):
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    market = yf.download(
        MARKET_TICKERS,
        start=start, end=end,
        auto_adjust=True, progress=False, group_by='ticker'
    )
    if isinstance(market.columns, pd.MultiIndex):
        market_close = market.xs('Close', axis=1, level=1)
    else:
        market_close = market[['Close']]

    market_close.columns = ['SPX', 'VIX']
    
    with open(cache_path, 'wb') as f:
        pickle.dump(market_close, f)
    return market_close


def download_risk_free_rate(start, end, cache_path='data/rf_rate.pkl'):
    """
    Download 3-Month T-Bill rate from FRED as daily risk-free rate proxy.
    Converts annualised % to daily decimal.
    """
    if os.path.exists(cache_path):
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    try:
        rf = web.DataReader('DGS3MO', 'fred', start, end)
        rf.columns = ['rf_annual_pct']
        rf['rf_daily'] = rf['rf_annual_pct'] / 100 / 252
        rf = rf.ffill()   # FRED has gaps on weekends/holidays
        with open(cache_path, 'wb') as f:
            pickle.dump(rf, f)
        return rf
    except Exception as e:
        print(f"FRED download failed: {e}")
        print("Using constant 4% annual risk-free rate as fallback.")
        dates = pd.date_range(start, end, freq='B')
        rf = pd.DataFrame({
            'rf_annual_pct': 4.0,
            'rf_daily'     : 4.0 / 100 / 252
        }, index=dates)
        return rf


# ── Run downloads ─────────────────────────────────────────────────────────────
prices    = download_price_data(SP100_TICKERS, START_DATE, END_DATE)
market    = download_market_data(START_DATE, END_DATE)
rf_rate   = download_risk_free_rate(START_DATE, END_DATE)

print(f"\nPrices  shape : {prices.shape}")
print(f"Market  shape : {market.shape}")
print(f"RF rate shape : {rf_rate.shape}")
print(f"\nPrice date range: {prices.index[0].date()} → {prices.index[-1].date()}")


## 3. Data Cleaning & Quality Assurance

### Why this matters for a quant
Garbage in = garbage out. A model built on uncleaned price data will have:
- Inflated volatility estimates from data errors
- Incorrect covariance matrices (critical for Markowitz optimisation)
- Momentum signals polluted by non-economic price jumps

We apply the following cleaning steps systematically and document every decision.


In [ ]:
def clean_price_data(prices, min_history_frac=0.80, max_zero_return_frac=0.05):
    """
    Comprehensive price data cleaning pipeline.
    
    Steps:
      1. Remove tickers with insufficient history
      2. Forward-fill short gaps (≤5 days) — weekends, holidays
      3. Drop tickers with excessive zero-return days (data errors)
      4. Detect and flag price anomalies (>10σ single-day moves)
      5. Document all removals
    """
    report = {}
    original_tickers = prices.columns.tolist()
    report['original_count'] = len(original_tickers)

    # ── Step 1: Minimum history filter ──────────────────────────────────────
    min_obs = int(prices.shape[0] * min_history_frac)
    valid_count = prices.notna().sum()
    tickers_to_drop_history = valid_count[valid_count < min_obs].index.tolist()
    prices = prices.drop(columns=tickers_to_drop_history)
    report['dropped_insufficient_history'] = tickers_to_drop_history

    # ── Step 2: Forward-fill short gaps ─────────────────────────────────────
    prices = prices.ffill(limit=5)

    # ── Step 3: Zero-return filter ───────────────────────────────────────────
    daily_rets = prices.pct_change()
    zero_return_frac = (daily_rets == 0).sum() / len(daily_rets)
    tickers_to_drop_zero = zero_return_frac[zero_return_frac > max_zero_return_frac].index.tolist()
    prices = prices.drop(columns=tickers_to_drop_zero)
    report['dropped_excessive_zeros'] = tickers_to_drop_zero

    # ── Step 4: Anomaly detection — flag but keep ────────────────────────────
    log_rets = np.log(prices / prices.shift(1))
    rolling_std = log_rets.rolling(60).std()
    z_scores    = log_rets / rolling_std
    anomalies   = (np.abs(z_scores) > 10)
    n_anomalies = anomalies.sum().sum()
    report['price_anomalies_flagged'] = int(n_anomalies)

    # ── Step 5: Drop remaining NaNs after ffill ──────────────────────────────
    prices = prices.dropna(how='all', axis=0)
    prices = prices.dropna(how='all', axis=1)

    report['final_count'] = len(prices.columns)
    report['final_obs']   = len(prices)
    return prices, report


prices_clean, cleaning_report = clean_price_data(prices)

print("=" * 55)
print("DATA CLEANING REPORT")
print("=" * 55)
print(f"Original tickers       : {cleaning_report['original_count']}")
print(f"Dropped (history)      : {len(cleaning_report['dropped_insufficient_history'])}")
if cleaning_report['dropped_insufficient_history']:
    print(f"  → {cleaning_report['dropped_insufficient_history']}")
print(f"Dropped (zero returns) : {len(cleaning_report['dropped_excessive_zeros'])}")
if cleaning_report['dropped_excessive_zeros']:
    print(f"  → {cleaning_report['dropped_excessive_zeros']}")
print(f"Anomalies flagged      : {cleaning_report['price_anomalies_flagged']}")
print(f"Final universe         : {cleaning_report['final_count']} stocks")
print(f"Observations           : {cleaning_report['final_obs']} trading days")
print("=" * 55)

TICKERS = prices_clean.columns.tolist()


## 4. Return Engineering

### Log Returns vs Simple Returns — Quant Perspective

We use **log returns** throughout this project:

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

**Why log returns?**
1. **Additivity over time**: $r_{0→T} = \sum_{t=1}^{T} r_t$ — critical for multi-period analysis
2. **Symmetry**: A 50% loss followed by 100% gain ≠ 0 in simple returns, but correctly = 0 in log returns
3. **Normality**: Log returns are closer to normally distributed (though fat tails remain)
4. **Standard in quant finance**: All volatility models (GARCH, realised vol) assume log returns


In [ ]:
# ── Log returns ───────────────────────────────────────────────────────────────
log_returns = np.log(prices_clean / prices_clean.shift(1)).dropna()

# ── Simple returns (kept for comparison) ──────────────────────────────────────
simple_returns = prices_clean.pct_change().dropna()

# ── Market log returns ────────────────────────────────────────────────────────
market_clean = market.reindex(log_returns.index, method='ffill').dropna()
spx_returns  = np.log(market_clean['SPX'] / market_clean['SPX'].shift(1)).dropna()
vix          = market_clean['VIX'].reindex(log_returns.index, method='ffill')

# ── Align all data to common index ────────────────────────────────────────────
common_idx  = log_returns.index.intersection(spx_returns.index)
log_returns = log_returns.loc[common_idx]
spx_returns = spx_returns.loc[common_idx]
vix         = vix.loc[common_idx]

print(f"Log returns shape     : {log_returns.shape}")
print(f"Date range            : {log_returns.index[0].date()} → {log_returns.index[-1].date()}")
print(f"\nSample log returns (AAPL, first 5 rows):")
print(log_returns['AAPL'].head().round(6))

# ── Save cleaned data ─────────────────────────────────────────────────────────
log_returns.to_parquet('data/log_returns.parquet')
prices_clean.to_parquet('data/prices_clean.parquet')
spx_returns.to_frame('SPX').to_parquet('data/spx_returns.parquet')
vix.to_frame('VIX').to_parquet('data/vix.parquet')
print("\nAll cleaned data saved to data/ directory.")


## 5. Missing Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Missing data heatmap (sample of 30 tickers for visibility)
sample_tickers = TICKERS[:30]
missing_matrix = prices_clean[sample_tickers].isna().astype(int)

axes[0].imshow(missing_matrix.T, aspect='auto', cmap='RdYlGn_r',
               interpolation='none', vmin=0, vmax=1)
axes[0].set_yticks(range(len(sample_tickers)))
axes[0].set_yticklabels(sample_tickers, fontsize=7)
n_ticks = 8
step = len(prices_clean) // n_ticks
axes[0].set_xticks(range(0, len(prices_clean), step))
axes[0].set_xticklabels(
    [prices_clean.index[i].strftime('%Y') for i in range(0, len(prices_clean), step)],
    rotation=45, fontsize=8
)
axes[0].set_title('Missing Data Heatmap (first 30 tickers)
Green=Present, Red=Missing')

# Missing % per ticker
missing_pct = (prices_clean.isna().sum() / len(prices_clean) * 100).sort_values(ascending=False)
top20_missing = missing_pct.head(20)
axes[1].barh(range(len(top20_missing)), top20_missing.values,
             color=[COLORS['negative'] if v > 5 else COLORS['accent'] if v > 1 else COLORS['positive']
                    for v in top20_missing.values])
axes[1].set_yticks(range(len(top20_missing)))
axes[1].set_yticklabels(top20_missing.index, fontsize=8)
axes[1].axvline(5, color='red', lw=1.5, linestyle='--', label='5% threshold')
axes[1].set_xlabel('Missing %')
axes[1].set_title('Missing Data % — Top 20 Tickers')
axes[1].legend()

plt.suptitle('Data Quality Assessment — Missing Values', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/01_missing_data.png', dpi=130, bbox_inches='tight')
plt.show()

print(f"Mean missing % across universe: {missing_pct.mean():.2f}%")
print(f"Tickers with >5% missing: {(missing_pct > 5).sum()}")


## 6. Price History Visualisation

In [ ]:
# Normalised price chart — set all to 100 at start for comparability
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Select representative tickers across sectors
showcase = {
    'Tech'    : ['AAPL', 'MSFT', 'NVDA', 'META'],
    'Finance' : ['JPM', 'GS', 'BAC'],
    'Energy'  : ['XOM', 'CVX'],
    'Consumer': ['AMZN', 'COST'],
}

# Normalised prices (rebased to 100)
prices_norm = (prices_clean / prices_clean.iloc[0]) * 100

sector_colors = ['#003087','#C6002B','#F5A623','#1a7a4a','#9b59b6']
ax = axes[0]
for (sector, tickers_s), col in zip(showcase.items(), sector_colors):
    for i, t in enumerate(tickers_s):
        if t in prices_norm.columns:
            lw = 2.5 if i == 0 else 1.2
            ls = '-' if i == 0 else '--'
            ax.plot(prices_norm.index, prices_norm[t],
                    label=f"{t} ({sector})", color=col, lw=lw, ls=ls, alpha=0.9)

ax.axvspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-04-15'),
           alpha=0.15, color='red', label='COVID Crash')
ax.axvspan(pd.Timestamp('2022-01-01'), pd.Timestamp('2022-10-01'),
           alpha=0.10, color='orange', label='2022 Bear Market')
ax.set_ylabel('Price (rebased to 100)')
ax.set_title('S&P 100 — Normalised Price History 2014–2025')
ax.legend(ncol=4, fontsize=8, loc='upper left')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# VIX overlay
ax2 = axes[1]
ax2.fill_between(vix.index, vix.values, 0,
                 where=(vix.values > 0), alpha=0.6, color=COLORS['secondary'], label='VIX')
ax2.axhline(20, color='orange', lw=1.5, ls='--', label='VIX=20 (elevated risk)')
ax2.axhline(30, color='red',    lw=1.5, ls='--', label='VIX=30 (stress)')
ax2.axhspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-04-15'), alpha=0.15, color='red')
ax2.axhspan(pd.Timestamp('2022-01-01'), pd.Timestamp('2022-10-01'), alpha=0.10, color='orange')
ax2.set_ylabel('VIX Level')
ax2.set_title('CBOE Volatility Index (VIX) 2014–2025 — Fear Gauge')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/02_price_history.png', dpi=130, bbox_inches='tight')
plt.show()


## 7. Market Regime Identification

A key quant insight: **models perform differently across market regimes**.
We define regimes using VIX levels and will use these throughout the project
to test whether ML volatility forecasting adds more value in stress regimes.

| Regime | VIX Level | Characteristic |
|--------|-----------|----------------|
| Calm | VIX < 15 | Low volatility, trending markets |
| Normal | 15 ≤ VIX < 25 | Average market conditions |
| Elevated | 25 ≤ VIX < 35 | Stress — correlations rise |
| Crisis | VIX ≥ 35 | Extreme — all strategies tested |


In [ ]:
# ── Regime classification ─────────────────────────────────────────────────────
def classify_regime(vix_level):
    if vix_level < 15:   return 'Calm'
    elif vix_level < 25: return 'Normal'
    elif vix_level < 35: return 'Elevated'
    else:                return 'Crisis'

regimes = vix.apply(classify_regime)
regime_colors = {'Calm':'#1a7a4a','Normal':'#F5A623','Elevated':'#e67e22','Crisis':'#C6002B'}

# ── Regime summary statistics ─────────────────────────────────────────────────
regime_stats = []
for regime in ['Calm', 'Normal', 'Elevated', 'Crisis']:
    mask       = (regimes == regime)
    spx_r      = spx_returns[mask]
    regime_stats.append({
        'Regime'          : regime,
        'Days'            : mask.sum(),
        'Pct of Sample'   : f"{mask.mean()*100:.1f}%",
        'SPX Mean Return' : f"{spx_r.mean()*252:.2%}",
        'SPX Ann Vol'     : f"{spx_r.std()*np.sqrt(252):.2%}",
        'Mean VIX'        : f"{vix[mask].mean():.1f}",
        'Max VIX'         : f"{vix[mask].max():.1f}",
    })
regime_df = pd.DataFrame(regime_stats)
print("MARKET REGIME SUMMARY")
print("=" * 70)
print(regime_df.to_string(index=False))

# ── Visualise regimes on SPX ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

spx_cum = (1 + spx_returns).cumprod()
axes[0].plot(spx_cum.index, spx_cum.values, color=COLORS['primary'], lw=1.5)

for regime, color in regime_colors.items():
    mask = (regimes == regime)
    axes[0].fill_between(spx_cum.index, spx_cum.min()*0.95, spx_cum.max()*1.05,
                         where=mask.values, alpha=0.15, color=color, label=regime)
axes[0].set_ylabel('SPX Cumulative Return'); axes[0].legend(ncol=4, fontsize=9)
axes[0].set_title('S&P 500 Cumulative Return with Market Regimes')

# Regime pie
regime_counts = regimes.value_counts().reindex(['Calm','Normal','Elevated','Crisis'])
axes[1].pie(regime_counts.values,
            labels=[f"{r}\n({c} days)" for r, c in zip(regime_counts.index, regime_counts.values)],
            colors=[regime_colors[r] for r in regime_counts.index],
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 10})
axes[1].set_title('Distribution of Market Regimes 2014–2025')

plt.tight_layout()
plt.savefig('data/03_market_regimes.png', dpi=130, bbox_inches='tight')
plt.show()

# Save regimes for downstream notebooks
regimes.to_frame('regime').to_parquet('data/market_regimes.parquet')
print("Regimes saved.")


## 8. Stationarity Testing — Augmented Dickey-Fuller

**Why stationarity matters for quant models:**
- GARCH models require stationary input (returns, not prices)
- Non-stationary features in ML models cause spurious patterns
- ADF test: H₀ = unit root present (non-stationary) → reject H₀ → stationary

We test: prices (non-stationary expected) and log returns (stationary expected).
This validates that our return engineering step was correct.


In [ ]:
def adf_test(series, name='', max_lags=20):
    """ADF test with automatic lag selection (AIC criterion)."""
    result = adfuller(series.dropna(), maxlag=max_lags, autolag='AIC')
    return {
        'Series'     : name,
        'ADF Stat'   : round(result[0], 4),
        'p-value'    : round(result[1], 4),
        'Lags Used'  : result[2],
        '1% Critical': round(result[4]['1%'], 4),
        '5% Critical': round(result[4]['5%'], 4),
        'Stationary' : '✓ YES' if result[1] < 0.05 else '✗ NO',
    }

# Test on sample of 10 tickers
sample_10 = TICKERS[:10]
adf_results = []

for ticker in sample_10:
    # Test price level
    adf_results.append(adf_test(prices_clean[ticker].dropna(), f'{ticker} (Price)'))
    # Test log return
    adf_results.append(adf_test(log_returns[ticker].dropna(),  f'{ticker} (Log Return)'))

adf_df = pd.DataFrame(adf_results)
print("AUGMENTED DICKEY-FULLER STATIONARITY TESTS")
print("=" * 75)
print(adf_df[['Series','ADF Stat','p-value','Stationary']].to_string(index=False))
print()
stat_returns = (adf_df[adf_df['Series'].str.contains('Log Return')]['Stationary'] == '✓ YES').mean()
stat_prices  = (adf_df[adf_df['Series'].str.contains('Price')]['Stationary']      == '✓ YES').mean()
print(f"Stationary (Log Returns) : {stat_returns:.0%}")
print(f"Stationary (Prices)      : {stat_prices:.0%}")
print()
print("Finding: Log returns are stationary (expected). Prices are non-stationary (unit root).")
print("→ Validates use of log returns for all downstream modelling.")


## 9. Final Dataset Summary & Save

In [ ]:
print("=" * 60)
print("PREPROCESSING COMPLETE — FINAL DATASET SUMMARY")
print("=" * 60)
print(f"Universe          : {len(TICKERS)} stocks (S&P 100)")
print(f"Date range        : {log_returns.index[0].date()} → {log_returns.index[-1].date()}")
print(f"Trading days      : {len(log_returns):,}")
print(f"Total observations: {log_returns.shape[0] * log_returns.shape[1]:,}")
print()
print("Return Statistics (cross-sectional mean):")
ann_ret = log_returns.mean() * 252
ann_vol = log_returns.std()  * np.sqrt(252)
print(f"  Mean annualised return  : {ann_ret.mean():.2%}")
print(f"  Mean annualised vol     : {ann_vol.mean():.2%}")
print(f"  Mean Sharpe (approx)    : {(ann_ret / ann_vol).mean():.3f}")
print()
print("Files saved:")
print("  data/log_returns.parquet")
print("  data/prices_clean.parquet")
print("  data/spx_returns.parquet")
print("  data/vix.parquet")
print("  data/market_regimes.parquet")

# Save ticker list
pd.Series(TICKERS).to_csv('data/tickers.csv', index=False, header=False)
print("  data/tickers.csv")
